# Prototyp – Geeignetheitsprüfung mit KI-Begründung

Der Bankberater gibt einen Kundennamen ein. Feste Regeln prüfen alle Produkte gegen das Kundenprofil und wählen bis zu drei Vorschläge aus. Ein kleines Open-Weight-Sprachmodell formuliert daraus eine kurze Begründung. Jeder Schritt landet im Audit-Log.

**Prinzip:** Regeln entscheiden, das Sprachmodell erklärt, der Bankberater entscheidet.

| Zelle | Inhalt |
|---|---|
| 1 | Setup |
| 2 | Kundendatenbank (fiktiv) |
| 3 | Produktdatenbank (fiktiv) |
| 4 | Regeln: Geeignetheit prüfen, Vorschläge auswählen |
| 5 | Sprachmodell laden (läuft in der eigenen Umgebung, kein externer Anbieter) |
| 6 | Bankberater fragt: Ergebnis, Begründung, Audit-Log |

Alle Daten sind erfunden. Bewusst nicht enthalten: echte Daten, Kundenkanal, Login, Evaluation.

In [ ]:
# ==========================================================
# ZELLE 1  Setup
# ==========================================================
import json, sqlite3, datetime
import pandas as pd

DB = "bank.db"
NACHHALTIGKEIT = {0: "keine", 1: "mittel", 2: "hoch"}
KENNTNISSE = {1: "Basis", 2: "Informiert", 3: "Fortgeschritten"}
VERLUST = {1: "gering", 2: "mittel", 3: "hoch (bis Totalverlust)"}
audit_log = []

def protokolliere(schritt, eingabe, ausgabe):
    audit_log.append({"zeit": datetime.datetime.now().isoformat(timespec="seconds"),
                      "schritt": schritt, "eingabe": eingabe, "ausgabe": ausgabe})

In [ ]:
# ==========================================================
# ZELLE 2  Kundendatenbank (fiktiv)
# ==========================================================
# nur die Felder, die für die Geeignetheit nötig sind (Datenminimierung)
# Skalen sind dieselben wie bei den Produkten, nur so funktioniert das Mapping
kunden = pd.DataFrame([
    # kundennr, name,          geburtsjahr, risiko 1-7, horizont_jahre, kenntnisse 1-3, verlust_tragbar 1-3, nachhaltigkeit 0-2
    ("K-1001", "Lena Berger",   1999, 4, 10, 2, 3, 2),
    ("K-1002", "Markus Weber",  1968, 2,  3, 1, 1, 0),
    ("K-1003", "Sabine Koch",   1982, 6, 15, 3, 3, 1),
    ("K-1004", "Jonas Schmitt", 1995, 5,  7, 2, 3, 0),
    ("K-1005", "Helga Braun",   1955, 1,  2, 1, 1, 1),
], columns=["kundennr", "name", "geburtsjahr", "risiko", "horizont_jahre", "kenntnisse", "verlust_tragbar", "nachhaltigkeit"])

with sqlite3.connect(DB) as con:
    kunden.to_sql("kunden", con, if_exists="replace", index=False)
kunden

,kundennr,name,geburtsjahr,risiko,horizont_jahre,kenntnisse,verlust_tragbar,nachhaltigkeit
0,K-1001,Lena Berger,1999,4,10,2,3,2
1,K-1002,Markus Weber,1968,2,3,1,1,0
2,K-1003,Sabine Koch,1982,6,15,3,3,1
3,K-1004,Jonas Schmitt,1995,5,7,2,3,0
4,K-1005,Helga Braun,1955,1,2,1,1,1


In [ ]:
# ==========================================================
# ZELLE 3  Produktdatenbank (fiktiv)
# ==========================================================
# risiko = Risikoklasse 1-7 aus dem Basisinformationsblatt
# haltedauer = empfohlene Mindesthaltedauer in Jahren
# nachhaltigkeit: Fonds/ETF nach SFDR (Art. 6 = 0, Art. 8 = 1, Art. 9 = 2), Aktien/Anleihen nach ESG-Rating
# hausmeinung = Einschätzung aus dem Research der Bank, keine Prognose der KI
produkte = pd.DataFrame([
    # id,   name,                              art,          risiko, haltedauer, kenntnisse, verlust, nachhaltigkeit, hausmeinung
    ("P01", "Geldmarktfonds Euro",              "Fonds",      1, 1, 1, 1, 1, "positiv"),
    ("P02", "Staatsanleihe Euroraum 2028",      "Anleihe",    2, 2, 1, 1, 0, "neutral"),
    ("P03", "Pfandbrief Nordbank 2030",         "Anleihe",    2, 4, 1, 2, 0, "positiv"),
    ("P04", "Grüne Anleihe Windpark 2032",      "Anleihe",    3, 6, 2, 2, 2, "positiv"),
    ("P05", "Unternehmensanleihe Alpina 2029",  "Anleihe",    3, 3, 2, 2, 1, "positiv"),
    ("P06", "Hochzinsanleihe Velox 2030",       "Anleihe",    5, 4, 3, 3, 0, "neutral"),
    ("P07", "Rentenfonds Europa",               "Fonds",      2, 3, 1, 2, 1, "positiv"),
    ("P08", "Mischfonds Ausgewogen",            "Fonds",      3, 5, 1, 2, 1, "positiv"),
    ("P09", "Mischfonds Nachhaltig",            "Fonds",      3, 5, 1, 2, 2, "positiv"),
    ("P10", "Aktien-ETF Welt",                  "ETF",        4, 7, 1, 3, 0, "positiv"),
    ("P11", "Aktien-ETF Klima",                 "ETF",        4, 7, 1, 3, 2, "positiv"),
    ("P12", "Nordwind Energie AG",              "Aktie",      4, 5, 2, 3, 2, "negativ"),
    ("P13", "Ferrox Stahl AG",                  "Aktie",      5, 5, 2, 3, 0, "negativ"),
    ("P14", "Quantix Tech AG",                  "Aktie",      6, 7, 2, 3, 1, "positiv"),
    ("P15", "Discount-Zertifikat Tech-Index",   "Zertifikat", 5, 2, 3, 3, 0, "neutral"),
], columns=["id", "name", "art", "risiko", "haltedauer", "kenntnisse", "verlust", "nachhaltigkeit", "hausmeinung"])

with sqlite3.connect(DB) as con:
    produkte.to_sql("produkte", con, if_exists="replace", index=False)
produkte

,id,name,art,risiko,haltedauer,kenntnisse,verlust,nachhaltigkeit,hausmeinung
0,P01,Geldmarktfonds Euro,Fonds,1,1,1,1,1,positiv
1,P02,Staatsanleihe Euroraum 2028,Anleihe,2,2,1,1,0,neutral
2,P03,Pfandbrief Nordbank 2030,Anleihe,2,4,1,2,0,positiv
3,P04,Grüne Anleihe Windpark 2032,Anleihe,3,6,2,2,2,positiv
4,P05,Unternehmensanleihe Alpina 2029,Anleihe,3,3,2,2,1,positiv
5,P06,Hochzinsanleihe Velox 2030,Anleihe,5,4,3,3,0,neutral
6,P07,Rentenfonds Europa,Fonds,2,3,1,2,1,positiv
7,P08,Mischfonds Ausgewogen,Fonds,3,5,1,2,1,positiv
8,P09,Mischfonds Nachhaltig,Fonds,3,5,1,2,2,positiv
9,P10,Aktien-ETF Welt,ETF,4,7,1,3,0,positiv


In [ ]:
# ==========================================================
# ZELLE 4  Regeln: hier fällt die Entscheidung, ohne KI
# ==========================================================
def lese(sql, parameter=()):
    # Datenbank nur lesend öffnen, Eingaben als Parameter statt im SQL-Text
    with sqlite3.connect(f"file:{DB}?mode=ro", uri=True) as con:
        return pd.read_sql(sql, con, params=parameter)

def kunde_suchen(name):
    df = lese("SELECT * FROM kunden WHERE lower(name) = lower(?) OR kundennr = ?", (name.strip(), name.strip()))
    return None if df.empty else df.iloc[0]

def geeignetheit_pruefen(k):
    # Stufe A, Pflicht nach MiFID II: feste Regeln, gleiche Skalen bei Kunde und Produkt
    geeignet, aussortiert = [], []
    for _, p in lese("SELECT * FROM produkte").iterrows():
        gruende = []
        if p.risiko > k.risiko:                 gruende.append(f"Risikoklasse {p.risiko} über Grenze {k.risiko}")
        if p.haltedauer > k.horizont_jahre:     gruende.append(f"Haltedauer {p.haltedauer} J. länger als Horizont {k.horizont_jahre} J.")
        if p.kenntnisse > k.kenntnisse:         gruende.append(f"verlangt Kenntnisse '{KENNTNISSE[p.kenntnisse]}'")
        if p.verlust > k.verlust_tragbar:       gruende.append(f"Verlustrisiko {VERLUST[p.verlust]} zu hoch")
        if p.nachhaltigkeit < k.nachhaltigkeit: gruende.append(f"Nachhaltigkeit {NACHHALTIGKEIT[p.nachhaltigkeit]}, Kunde wünscht {NACHHALTIGKEIT[k.nachhaltigkeit]}")
        (aussortiert if gruende else geeignet).append((p, gruende))
    return geeignet, aussortiert

def vorschlaege_auswaehlen(k, geeignet):
    # Stufe B, Vorgabe der Bank: negative Hausmeinung raus, positiv vor neutral, dann Nähe zur Risikobereitschaft
    kandidaten = [p for p, _ in geeignet if p.hausmeinung != "negativ"]
    kandidaten.sort(key=lambda p: (p.hausmeinung != "positiv", k.risiko - p.risiko))
    return kandidaten[:3], [p["name"] for p, _ in geeignet if p.hausmeinung == "negativ"]

# Test ohne KI
k = kunde_suchen("Lena Berger")
geeignet, aussortiert = geeignetheit_pruefen(k)
print(f"{k['name']}: {len(geeignet)} von {len(geeignet) + len(aussortiert)} Produkten geeignet")

Lena Berger: 4 von 15 Produkten geeignet


In [ ]:
# ==========================================================
# ZELLE 5  Sprachmodell laden (erklärt nur, entscheidet nichts)
# ==========================================================
# kleines Open-Weight-Modell, läuft in der eigenen Umgebung, kein externer Anbieter
# mit GPU (Laufzeit -> Laufzeittyp ändern -> T4-GPU) etwas größer, ohne GPU das kleinste
llm = None
try:
    import torch
    from transformers import pipeline
    modell_id = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"
    llm = pipeline("text-generation", model=modell_id, device_map="auto")
    print("Sprachmodell geladen:", modell_id)
except Exception as fehler:
    modell_id = None
    print("Kein Sprachmodell, Begründung kommt aus Textbausteinen:", " ".join(str(fehler).split())[:150])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Sprachmodell geladen: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
# ==========================================================
# ZELLE 6  Bankberater fragt
# ==========================================================
# zum Ausprobieren: "Lena Berger", "Markus Weber" oder "Jonas Schmitt"
NAME = "Markus Weber"

k = kunde_suchen(NAME)
protokolliere("kunde_suchen", NAME, None if k is None else k.kundennr)
if k is None:
    raise SystemExit(f"Kein Kunde gefunden für '{NAME}'")

geeignet, aussortiert = geeignetheit_pruefen(k)
protokolliere("geeignetheit_pruefen", k.kundennr, [p["name"] for p, _ in geeignet])
vorschlaege, wegen_hausmeinung = vorschlaege_auswaehlen(k, geeignet)
protokolliere("vorschlaege_auswaehlen", k.kundennr, [p["name"] for p in vorschlaege])

# Tabelle für den Bankberater: Risikoklasse und Nachhaltigkeit immer neben den Werten des Kunden
tabelle = pd.DataFrame([{
    "Produkt": p["name"], "Art": p.art,
    "Risikoklasse": f"{p.risiko} (Kunde bis {k.risiko})",
    "Nachhaltigkeit": f"{NACHHALTIGKEIT[p.nachhaltigkeit]} (Kunde: {NACHHALTIGKEIT[k.nachhaltigkeit]})",
    "Hausmeinung": p.hausmeinung} for p in vorschlaege])

# Fakten für die Begründung, nur aus den Regeln, in klaren Worten
stufe = lambda r: "niedrige" if r <= 2 else ("mittlere" if r <= 4 else "hohe")
fakten = (f"{k['name']} hat eine {stufe(k.risiko)} Risikobereitschaft (bis Risikoklasse {k.risiko}), "
          f"einen Anlagehorizont von {k.horizont_jahre} Jahren und den Nachhaltigkeitswunsch '{NACHHALTIGKEIT[k.nachhaltigkeit]}'. "
          f"Diese Produkte sind geeignet und werden vorgeschlagen: "
          f"{', '.join(p['name'] + ' (Risikoklasse ' + str(p.risiko) + ')' for p in vorschlaege)}. "
          f"{len(aussortiert)} andere Produkte passen nicht zum Profil"
          + (f". Geeignet, aber wegen negativer Hausmeinung nicht vorgeschlagen: {', '.join(wegen_hausmeinung)}." if wegen_hausmeinung else "."))

def regel_check(text):
    # Regel-Check auf die KI-Ausgabe: alle Vorschläge genannt, nichts Widersprüchliches
    t = text.lower()
    namen_ok = all(p["name"].lower() in t for p in vorschlaege)
    falsche_stufe = any(f"{s} risikobereitschaft" in t for s in ("niedrige", "mittlere", "hohe") if s != stufe(k.risiko))
    verboten = any(w in t for w in ("passen nicht zu", "passten nicht", "nicht geeignet", "rendite"))
    return namen_ok and not falsche_stufe and not verboten

begruendung, quelle = fakten, "Textbaustein"
if llm is not None:
    try:
        nachricht = [{"role": "user", "content": "Fasse diese Fakten für einen Bankberater in 2 sachlichen Sätzen auf Deutsch zusammen. "
                      "Ohne Einleitung, ohne Aufzählung. Übernimm Produktnamen und Zahlen wörtlich, erfinde nichts:\n" + fakten}]
        text = llm(nachricht, max_new_tokens=150, do_sample=False)[0]["generated_text"][-1]["content"].strip()
        if regel_check(text):
            begruendung, quelle = text, modell_id
        else:
            quelle = f"Textbaustein (KI-Text von {modell_id} hat den Regel-Check nicht bestanden)"
            print("KI-Text verworfen:\n" + text + "\n")
    except Exception as fehler:
        print("Sprachmodell-Fehler, Textbaustein wird genutzt:", " ".join(str(fehler).split())[:150])
protokolliere("begruendung", quelle, begruendung)
audit_log[-1]["freigabe_bankberater"] = "offen"   # Human-in-the-Loop: erst der Bankberater gibt frei

print(f"VORSCHLÄGE FÜR {k['name'].upper()} – Freigabe durch den Bankberater")
display(tabelle)
print("\nBEGRÜNDUNG (" + quelle + "):\n" + begruendung)

with open("audit_log.json", "w", encoding="utf-8") as f:
    json.dump(audit_log, f, ensure_ascii=False, indent=2, default=str)
print("\nAUDIT-LOG")
display(pd.DataFrame(audit_log)[["zeit", "schritt", "eingabe"]])

VORSCHLÄGE FÜR MARKUS WEBER – Freigabe durch den Bankberater


,Produkt,Art,Risikoklasse,Nachhaltigkeit,Hausmeinung
0,Geldmarktfonds Euro,Fonds,1 (Kunde bis 2),mittel (Kunde: keine),positiv
1,Staatsanleihe Euroraum 2028,Anleihe,2 (Kunde bis 2),keine (Kunde: keine),neutral



BEGRÜNDUNG (Qwen/Qwen2.5-0.5B-Instruct):
Markus Weber hat eine niedrige Risikobereitschaft (bis Risikoklasse 2) und einen Anlagehorizont von 3 Jahren. Er behält keine Nachhaltigkeit als Priorität. Diese Produkte sind geeignet und werden vorgeschlagen: Geldmarktfonds Euro (Risikoklasse 1), Staatsanleihe Euroraum 2028 (Risikoklasse 2). 13 andere Produkte passt nicht zu seinem Profil.

AUDIT-LOG


,zeit,schritt,eingabe
0,2026-10-29T12:26:14,kunde_suchen,Lena Berger
1,2026-10-29T12:26:14,geeignetheit_pruefen,K-1001
2,2026-10-29T12:26:14,vorschlaege_auswaehlen,K-1001
3,2026-10-29T12:27:00,begruendung,Qwen/Qwen2.5-0.5B-Instruct
4,2026-10-29T12:28:31,kunde_suchen,Markus Weber
5,2026-10-29T12:28:31,geeignetheit_pruefen,K-1002
6,2026-10-29T12:28:31,vorschlaege_auswaehlen,K-1002
7,2026-10-29T12:29:06,begruendung,Qwen/Qwen2.5-0.5B-Instruct
